In [0]:
#Leitura camada bronze
df_movimento = spark.table("bronze.movimento_diaria_05hs")

In [0]:
from pyspark.sql import functions as F

df_silver_movimento = (
  df_movimento.select(
    F.col("id").cast("long").alias("id"),
    F.col("id_cartao").cast("long").alias("id_cartao"),
    F.col("vlr_transacao").cast("decimal(18,2)").alias("vlr_transacao"),
    F.initcap(F.trim(F.col("des_transacao"))).alias("des_transacao"),
    F.col("data_movimento").cast("timestamp").alias("data_movimento"),
    F.current_timestamp().alias("dt_ingestao_silver")
  )
  .dropDuplicates(["id"])
)



In [0]:
# Qualidade dos dados
erros = []  # Lista para armazenar erros de qualidade identificados

# Validação: vlr_transacao inválido
if df_silver_movimento.filter((F.col("vlr_transacao").isNull()) | (F.col("vlr_transacao") <= 0)).count() > 0:
    erros.append("vlr_transacao inválido")

# Validação: id_cartao nulo
if df_silver_movimento.filter(F.col("id_cartao").isNull()).count() > 0:
    erros.append("id_cartao nulo")

# Validação: des_transacao nulo ou vazio
if df_silver_movimento.filter((F.col("des_transacao").isNull()) | (F.col("des_transacao") == "")).count() > 0:
    erros.append("des_transacao nulo ou vazio")

# Validação: data_movimento nulo
if df_silver_movimento.filter(F.col("data_movimento").isNull()).count() > 0:
    erros.append("data_movimento nulo")

# Validação: data_movimento futura
if df_silver_movimento.filter(F.col("data_movimento") > F.current_timestamp()).count() > 0:
    erros.append("data_movimento futura")

# Se houver erros, imprime cada erro e lança uma exceção caso necessario
if erros:
    for erro in erros:
        print(f"Falha qualidade movimento_silver: {erro}")
    raise Exception("Falha qualidade movimento_silver: " + " | ".join(erros))

In [0]:
#Salvando base na camada silver e modo overwrite
(df_silver_movimento.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("silver.movimento_diaria_05hs"))